# 🎙️ F5-TTS: Zero-Shot Voice Cloning & High-Fidelity Audio Studio
### 100% Free, Zero Hugging Face Login & Zero Token Required

This is a standalone, production-ready Kaggle notebook for **[F5-TTS (Fairly Fast and Flexible Flow-Matching TTS)](https://github.com/SWivid/F5-TTS)**.

### ⚡ Quick Start Checklist:
1. Open Kaggle -> Create **New Notebook** (`File -> New Notebook`).
2. In the right sidebar under **Settings**:
   - **Accelerator:** Choose `GPU T4 x2` or `GPU P100`.
   - **Internet:** Toggle to `Internet On` (required for pip install & model weights).
3. Run each cell sequentially below.

---

## ⚙️ Cell 1: Check GPU Acceleration
Verifies NVIDIA GPU detection and CUDA availability on Kaggle.

In [ ]:
import torch
!nvidia-smi

print(f"\n✅ PyTorch Version: {torch.__version__}")
print(f"✅ CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"✅ Active GPU: {torch.cuda.get_device_name(0)}")
    print(f"✅ Total VRAM: {round(torch.cuda.get_device_properties(0).total_memory / (1024**3), 2)} GB")
else:
    raise SystemError("⚠️ GPU not detected! Please turn on GPU in Kaggle right sidebar: Settings -> Accelerator -> GPU T4 x2")

## 📦 Cell 2: Install F5-TTS, FFmpeg & Audio Tooling
Installs official `f5-tts`, `soundfile`, `ffmpeg`, `torchaudio`, `gradio`, and tunneling dependencies.

In [ ]:
import nest_asyncio
import uvicorn
import base64
import io
import time
import subprocess
import re
import os
from fastapi import FastAPI, HTTPException
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel, Field
from typing import Optional

nest_asyncio.apply()

# ---------------------------------------------------------------------------
# 1. Initialize FastAPI Application
# ---------------------------------------------------------------------------
app = FastAPI(title="F5-TTS Audio Engine API", version="1.0.0")

app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_credentials=True,
    allow_methods=["*"],
    allow_headers=["*"],
)

class SynthesizePayload(BaseModel):
    text: str = Field(..., description="Text to synthesize into speech")
    speed: Optional[float] = Field(1.0, description="Speech speed factor (0.8 - 1.4)")
    ref_audio_b64: Optional[str] = Field(None, description="Optional custom voice reference in base64")
    ref_text: Optional[str] = Field(None, description="Optional reference transcript")

@app.get("/health")
def health_check():
    return {"status": "ready", "engine": "F5-TTS_v1_Base", "cuda": torch.cuda.is_available()}

@app.post("/api/tts/synthesize")
def api_synthesize(payload: SynthesizePayload):
    t0 = time.time()
    ref_file = SAMPLE_VOICES["doctor_narrator"]["path"]
    ref_tx = SAMPLE_VOICES["doctor_narrator"]["ref_text"]
    
    if payload.ref_audio_b64:
        custom_path = f"temp_ref_{int(time.time())}.wav"
        with open(custom_path, "wb") as f:
            f.write(base64.b64decode(payload.ref_audio_b64))
        ref_file = custom_path
        if payload.ref_text:
            ref_tx = payload.ref_text
            
    try:
        wav, sr, _ = tts_engine.infer(
            ref_file=ref_file,
            ref_text=ref_tx,
            gen_text=payload.text,
            speed=payload.speed or 1.0
        )
        
        buf = io.BytesIO()
        sf.write(buf, wav, sr, format="WAV")
        b64_audio = base64.b64encode(buf.getvalue()).decode("utf-8")
        
        return {
            "status": "success",
            "latency_seconds": round(time.time() - t0, 2),
            "sample_rate": sr,
            "audio_format": "audio/wav",
            "audio_base64": b64_audio
        }
    except Exception as e:
        raise HTTPException(status_code=500, detail=str(e))

# ---------------------------------------------------------------------------
# 2. Connect Public URL Tunnel (Cloudflare or ngrok)
# ---------------------------------------------------------------------------
NGROK_TOKEN = ""  # 👈 Paste ngrok token here if you want to use ngrok

public_api_url = None

if NGROK_TOKEN.strip():
    from pyngrok import ngrok, conf
    conf.get_default().auth_token = NGROK_TOKEN.strip()
    tunnel = ngrok.connect(8001)
    public_api_url = tunnel.public_url
    print(f"\n🚀 [ngrok] Public F5-TTS REST API URL: {public_api_url}")
else:
    # Direct standalone cloudflared binary (No dpkg / sudo needed!)
    if not os.path.exists("./cloudflared"):
        print("⏳ Downloading standalone cloudflared binary...")
        !curl -fsSL -o ./cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
        !chmod +x ./cloudflared
    
    print("⏳ Starting Cloudflare public tunnel...")
    cf_proc = subprocess.Popen(["./cloudflared", "tunnel", "--url", "http://127.0.0.1:8001"], stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True)
    
    for _ in range(30):
        line = cf_proc.stderr.readline()
        if "trycloudflare.com" in line:
            match = re.search(r'https://[a-zA-Z0-9-]+\.trycloudflare\.com', line)
            if match:
                public_api_url = match.group(0)
                break
        time.sleep(0.5)
    
    if public_api_url:
        print(f"\n🚀 [Cloudflare] Public F5-TTS REST API URL: {public_api_url}")
    else:
        print("⚠️ Could not auto-detect Cloudflare URL. Please check cf_proc or use NGROK_TOKEN.")

if public_api_url:
    print(f"🔗 Test Health:     {public_api_url}/health")
    print(f"🔗 Test Synthesize: {public_api_url}/api/tts/synthesize\n")
    print("💡 Copy this URL into your web app frontend to play real-time AI voice!")

# ---------------------------------------------------------------------------
# 3. Start Uvicorn Server
# ---------------------------------------------------------------------------
uvicorn.run(app, host="0.0.0.0", port=8001)


## 🧠 Cell 3: Load F5-TTS Model & Vocoder Weights onto GPU
Loads pre-trained `F5TTS_v1_Base` flow-matching weights (public & ungated on Hugging Face, no token needed).

In [ ]:
import os
import time
from f5_tts.api import F5TTS

print("⏳ Loading F5-TTS model checkpoint onto GPU...")
t0 = time.time()

# Initialize F5-TTS model (uses CUDA GPU automatically)
tts_engine = F5TTS(model="F5TTS_v1_Base", device="cuda" if torch.cuda.is_available() else "cpu")

print(f"✅ F5-TTS Engine successfully loaded in {round(time.time() - t0, 2)}s!")
print(f"🚀 Model device: {tts_engine.device}")

## 🎙️ Cell 4: Download Sample Reference Voices
F5-TTS requires a clean 5-15 second reference audio clip (`.wav` or `.mp3`) and the exact text spoken in that clip.
This cell downloads a natural clear English reference voice to get you started immediately.

In [ ]:
import os
import urllib.request

os.makedirs("sample_voices", exist_ok=True)

# Sample reference voice
SAMPLE_VOICES = {
    "doctor_narrator": {
        "url": "https://raw.githubusercontent.com/SWivid/F5-TTS/main/src/f5_tts/infer/examples/basic/basic_ref_en.wav",
        "path": "sample_voices/basic_ref_en.wav",
        "ref_text": "Some have accepted the challenge, and found that there is more to it than meets the eye."
    }
}

for voice_name, vdata in SAMPLE_VOICES.items():
    if not os.path.exists(vdata["path"]):
        print(f"⏳ Downloading sample voice: {voice_name}...")
        try:
            urllib.request.urlretrieve(vdata["url"], vdata["path"])
            print(f"✅ Saved to {vdata['path']}")
        except Exception as e:
            print(f"⚠️ Download notice: {e}")

print("\n📁 Reference audio ready!")
print("💡 TIP: You can also upload your own audio file to Kaggle (/kaggle/working/) and use it as ref_audio!")

## 🔊 Cell 5: Generate Audio & Play in Notebook
Run inference to clone the voice and synthesize custom text. An interactive audio player is embedded directly in the output.

In [ ]:
import soundfile as sf
import IPython.display as ipd

# Customize the text to speak:
input_text = (
    "Welcome to LabLens AI. Your medical report has been successfully analyzed. "
    "Your systolic blood pressure reading is 140 millimeters of mercury, "
    "and your pulse rate is normal at 76 beats per minute. "
    "Please consult your primary care physician for personalized guidance."
)

ref_audio = SAMPLE_VOICES["doctor_narrator"]["path"]
ref_text = SAMPLE_VOICES["doctor_narrator"]["ref_text"]

print("⏳ Generating audio with F5-TTS...")
t0 = time.time()

wav, sr, _ = tts_engine.infer(
    ref_file=ref_audio,
    ref_text=ref_text,
    gen_text=input_text,
    speed=1.0
)

output_filename = "f5tts_output.wav"
sf.write(output_filename, wav, sr)
gen_time = round(time.time() - t0, 2)
duration = round(len(wav) / sr, 2)

print(f"✅ Audio generated in {gen_time}s! (Duration: {duration}s)")
print(f"📁 File saved as: {output_filename}")

# Play directly in Kaggle
ipd.Audio(output_filename)

## 🌐 Cell 6: Launch Interactive Gradio Web Studio (Public Share Link)
Launches the full interactive Gradio Studio with a **free public URL** (`https://*.gradio.live`).
You can:
- Record voice directly via microphone or upload an audio file.
- Enter reference text and target text.
- Adjust speech speed.
- Download the generated `.wav` file.

In [ ]:
import gradio as gr

def voice_clone_studio(ref_audio, ref_text, target_text, speed):
    if not ref_audio:
        return None, "⚠️ Please upload or record a reference voice audio clip (5-15 seconds)."
    if not target_text or not target_text.strip():
        return None, "⚠️ Please enter the text you want the voice to speak."
    
    t0 = time.time()
    wav, sr, _ = tts_engine.infer(
        ref_file=ref_audio,
        ref_text=ref_text.strip() if ref_text else "",
        gen_text=target_text.strip(),
        speed=float(speed)
    )
    out_path = f"studio_output_{int(time.time())}.wav"
    sf.write(out_path, wav, sr)
    elapsed = round(time.time() - t0, 2)
    duration = round(len(wav) / sr, 2)
    return out_path, f"✅ Generated in {elapsed}s (Audio Duration: {duration}s)"

ui = gr.Interface(
    fn=voice_clone_studio,
    inputs=[
        gr.Audio(type="filepath", label="Reference Voice (Upload WAV/MP3 or Record via Mic)"),
        gr.Textbox(label="Reference Audio Transcript (Words spoken in the reference audio)", placeholder="Type the exact words in the audio (or leave empty)..."),
        gr.Textbox(label="Text to Speak (Target text)", lines=4, placeholder="Enter the text you want spoken..."),
        gr.Slider(minimum=0.5, maximum=2.0, value=1.0, step=0.1, label="Speech Speed Multiplier")
    ],
    outputs=[
        gr.Audio(label="Synthesized Audio Output", type="filepath"),
        gr.Textbox(label="Generation Details & Latency")
    ],
    title="🎙️ F5-TTS: Voice Cloning & Audio Studio",
    description="High-fidelity zero-shot text-to-speech with flow matching. Upload a 5-15s audio clip of any voice to clone it instantly!"
)

# Launch with public URL
ui.launch(share=True, debug=False)

## ⚡ Cell 7: Launch FastAPI REST Server with ngrok Tunnel (Web App Integration)
Exposes a public HTTP REST API endpoint `/api/tts/synthesize` so external apps, frontends, or agents can request audio synthesis programmatically.

In [ ]:
import nest_asyncio
import uvicorn
import base64
import io
from fastapi import FastAPI, HTTPException
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel, Field
from typing import Optional
from pyngrok import ngrok

nest_asyncio.apply()

app = FastAPI(title="F5-TTS Audio Engine API", version="1.0.0")

app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_credentials=True,
    allow_methods=["*"],
    allow_headers=["*"],
)

class SynthesizePayload(BaseModel):
    text: str = Field(..., description="Text to synthesize")
    speed: Optional[float] = Field(1.0, description="Speed (0.8 - 1.5)")
    ref_audio_b64: Optional[str] = Field(None, description="Optional base64 reference audio")
    ref_text: Optional[str] = Field(None, description="Optional transcript of reference audio")

@app.get("/health")
def health_check():
    return {"status": "ready", "engine": "F5-TTS_v1_Base", "cuda": torch.cuda.is_available()}

@app.post("/api/tts/synthesize")
def api_synthesize(payload: SynthesizePayload):
    t0 = time.time()
    ref_file = SAMPLE_VOICES["doctor_narrator"]["path"]
    ref_tx = SAMPLE_VOICES["doctor_narrator"]["ref_text"]
    
    if payload.ref_audio_b64:
        custom_path = f"temp_ref_{int(time.time())}.wav"
        with open(custom_path, "wb") as f:
            f.write(base64.b64decode(payload.ref_audio_b64))
        ref_file = custom_path
        if payload.ref_text:
            ref_tx = payload.ref_text
            
    try:
        wav, sr, _ = tts_engine.infer(
            ref_file=ref_file,
            ref_text=ref_tx,
            gen_text=payload.text,
            speed=payload.speed or 1.0
        )
        
        buf = io.BytesIO()
        sf.write(buf, wav, sr, format="WAV")
        b64_audio = base64.b64encode(buf.getvalue()).decode("utf-8")
        
        return {
            "status": "success",
            "latency_seconds": round(time.time() - t0, 2),
            "sample_rate": sr,
            "audio_format": "audio/wav",
            "audio_base64": b64_audio
        }
    except Exception as e:
        raise HTTPException(status_code=500, detail=str(e))

# Connect ngrok tunnel on port 8001
tunnel = ngrok.connect(8001)
print(f"\n🚀 Public F5-TTS REST API URL: {tunnel.public_url}")
print(f"🔗 Test Health: {tunnel.public_url}/health")
print(f"🔗 Test Synthesize: {tunnel.public_url}/api/tts/synthesize\n")

uvicorn.run(app, host="0.0.0.0", port=8001)

## 💾 Cell 8: Batch Audio Synthesis & ZIP Download
Generate multiple voice clips and package them into a downloadable `.zip` file.

In [ ]:
import zipfile

test_phrases = [
    ("greeting.wav", "Hello, welcome to Samjho Health. Let's review your report."),
    ("normal_finding.wav", "All your metabolic panel markers are within normal limits."),
    ("doctor_advice.wav", "Please follow up with your doctor regarding blood pressure monitoring.")
]

os.makedirs("batch_output", exist_ok=True)

for filename, phrase in test_phrases:
    print(f"⏳ Generating {filename}...")
    w, s, _ = tts_engine.infer(
        ref_file=SAMPLE_VOICES["doctor_narrator"]["path"],
        ref_text=SAMPLE_VOICES["doctor_narrator"]["ref_text"],
        gen_text=phrase
    )
    sf.write(os.path.join("batch_output", filename), w, s)

# Create zip archive
zip_path = "f5tts_batch_audio.zip"
with zipfile.ZipFile(zip_path, 'w') as zf:
    for root, _, files in os.walk("batch_output"):
        for f in files:
            zf.write(os.path.join(root, f), f)

print(f"\n✅ Batch generation complete! Zip file saved at: {zip_path}")